## Fine tuning Open AI GPT-3.5  turbo model and job creation

In [ ]:
from openai import OpenAI
import openai
import os
from google.colab import userdata
openai.api_key = userdata.get('openai_api_key')
file-training_id = userdata.get('file-training_id')
file-validtion_id = userdata.get('file-validtion_id')
finetune_model_id = userdata.get('finetune_model_id')

In [ ]:
client = OpenAI(api_key="OPENAI_API_KEY")

In [ ]:
try:
    response = client.files.create(
        file=open("test.jsonl", "rb"),
        purpose="fine-tune"
    )
    print(response.id)
except Exception as e:
    print(f"Error: {e}")

In [ ]:
client.files.list()

In [ ]:
client.files.retrieve(file-validtion_id)

In [ ]:
client.fine_tuning.jobs.create(model = "gpt-3.5-turbo",
                               training_file=file-training_id,
                               hyperparameters={
                                   "n_epochs":1
                               },
                               validation_file=file-validtion_id)

In [ ]:
client.fine_tuning.jobs.list(limit=10)

In [ ]:
client.fine_tuning.jobs.retrieve(file-validtion_id)

In [ ]:
client.fine_tuning.jobs.list_events(fine_tuning_job_id="ftjob-ku7ytdtAEdx5jxthRANQTqr0",limit=5)

In [ ]:
completion = client.chat.completions.create(
  model="ft:gpt-3.5-turbo-0125:bis::9tt19es9",
  messages=[
    {"role": "user", "content": "Classify the articles into these categories: business, entertainment, politics, sport, tech. A new mobile phone is launched"}
  ]
)
print(completion.choices[0].message)

## Fine tune model evaluation

In [ ]:
from openai import OpenAI
import pandas as pd

In [ ]:
df = pd.read_csv('test.csv', encoding='unicode_escape')
df

In [ ]:
labels = df.iloc[:,1].tolist()
texts = df.iloc[:,0].tolist()

In [ ]:
texts = texts[450:]
labels = labels[450:]

In [ ]:
def inference_for_eval(text, m):
    completion = client.chat.completions.create(
    model=m,
    messages=[
        {"role": "system", "content": "You are a intelligent assistant designed to classify news articles into these categories: business, entertainment, politics, sport, tech."},
        {"role": "user", "content": text}
        ]
    )
    return completion.choices[0].message.content

In [ ]:
output = [inference_for_eval(text, finetune_model_id) for text in texts]

In [ ]:
print(output[:10])

In [ ]:
correct_classifications = sum(classification ==label for classification, label in zip(output,labels))

In [ ]:
total_classifications = len(labels)
print(total_classifications)

In [ ]:
accuracy_percentage = (correct_classifications/total_classifications)*100
print(f"Accuracy:{accuracy_percentage:.2f}%")